In [6]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import gc
from itertools import product
from sklearn.model_selection import train_test_split

# Global variable to track communication cost in bytes
communication_log = []

def compute_communication_cost(tensor, data_type_size=4):
    # data_type_size=4 for float32
    if isinstance(tensor, dict):
        size = sum(param.numel() for param in tensor.values())
    else:
        size = tensor.numel()
    return size * data_type_size


In [7]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [18]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, decentralized_train_n_epochs
from src.data_utils import create_batched_dataloaders, create_dataloader

In [ ]:
def log_and_update(user_model, neighbor_models, k=1000, quantize=False):
    global communication_log
    # Peer-to-peer model averaging (e.g., gossip-style)
    neighbor_avg = sum(neighbor_models) / len(neighbor_models)

    # Log outgoing communication
    for neighbor_model in neighbor_models:
        delta = user_model - neighbor_model
        sparse_delta = top_k_sparsify(delta, k)
        if quantize:
            sparse_delta = quantize_tensor(sparse_delta)
        cost = compute_communication_cost(sparse_delta)
        communication_log.append(cost)
    
    # Update user model
    user_model = 0.5 * user_model + 0.5 * neighbor_avg
    return user_model


def quantize_tensor(tensor, num_bits=8):
    qmin = 0.
    qmax = 2.**num_bits - 1.
    min_val, max_val = tensor.min(), tensor.max()
    scale = (max_val - min_val) / (qmax - qmin)
    quantized = ((tensor - min_val) / scale).round().clamp(qmin, qmax)
    return quantized


In [ ]:
class SparseFastNeighborDMF_NX:
    def __init__(self, num_users, num_items, latent_dim=10, lr=0.01,
                 alpha=0.01, beta=0.01, gamma=0.01, top_k=10):
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.lr = lr
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.top_k = top_k
        self.comm_cost = {u: {'sent': 0, 'recv': 0} for u in range(self.num_users)}

        self.user_factors = np.random.normal(scale=0.1, size=(num_users, latent_dim))
        self.common_item_factors = np.random.normal(scale=0.1, size=(num_items, latent_dim))
        self.personal_item_factors = {}  # (user_id, item_id) -> latent vector
        self.neighbors = {}

        # New logs
        self.epoch_times = []
        self.gradient_sharing_counts = []

    def build_neighbors_networkx(self, data, graph):
        if(sum(np.isnan(graph))):
            G = nx.Graph()
            item_user_map = {}

            for user_id, item_id, _ in data:
                item_user_map.setdefault(item_id, set()).add(user_id)

            for users in item_user_map.values():
                for u in users:
                    for v in users:
                        if u != v:
                            G.add_edge(u, v)
        else:
            G = graph

        for user_id in range(self.num_users):
            neighbors = list(G.neighbors(user_id)) if user_id in G else []
            if len(neighbors) > self.top_k:
                neighbors = np.random.choice(neighbors, size=self.top_k, replace=False)
            self.neighbors[user_id] = neighbors

    def train(self, data, val_data=None, epochs=100, patience=5, time_threshold=60):
        import time
    
        self.epoch_times = []
        self.gradient_sharing_counts = []
        self.train_losses = []
        self.val_losses = []
        epoch_summary = {}
    
        best_val_loss = float('inf')
        no_improve_count = 0
    
        for epoch in range(epochs):
            start_time = time.time()
            gradient_sharing_count = 0
            mse_train = 0.0
    
            np.random.shuffle(data)
            for user_id, item_id, rating in data:
                u = self.user_factors[user_id]
                p = self.common_item_factors[item_id]
                q_key = (user_id, item_id)
                if q_key not in self.personal_item_factors:
                    self.personal_item_factors[q_key] = np.random.normal(scale=0.1, size=self.latent_dim)
                q = self.personal_item_factors[q_key]
    
                pred = np.dot(u, p + q)
                error = rating - pred
                mse_train += error ** 2
    
                grad_u = -error * (p + q) + self.alpha * u
                grad_p = -error * u + self.beta * p
                grad_q = -error * u + self.gamma * q
    
                self.user_factors[user_id] -= self.lr * grad_u
                self.personal_item_factors[q_key] -= self.lr * grad_q
    
                for neighbor_id in self.neighbors.get(user_id, []):
                    self.common_item_factors[item_id] -= self.lr * grad_p / max(1, len(self.neighbors[user_id]))
                    gradient_sharing_count += 1
    
            epoch_time = time.time() - start_time
            rmse_train = np.sqrt(mse_train / len(data))
    
            # Validation loss
            if val_data is not None:
                mse_val = 0.0
                for user_id, item_id, rating in val_data:
                    pred = self.predict(user_id, item_id)
                    mse_val += (rating - pred) ** 2
                rmse_val = np.sqrt(mse_val / len(val_data))
            else:
                rmse_val = None
    
            # Store logs
            self.epoch_times.append(epoch_time)
            self.gradient_sharing_counts.append(gradient_sharing_count)
            self.train_losses.append(rmse_train)
            self.val_losses.append(rmse_val)
            epoch_summary[epoch] = {
                'time': epoch_time,
                'gradient_sharing': gradient_sharing_count,
                'train_rmse': rmse_train,
                'val_rmse': rmse_val
            }
    
           
            # Early stopping rules
            if rmse_val is not None:
                if rmse_val < best_val_loss:
                    best_val_loss = rmse_val
                    no_improve_count = 0
                else:
                    no_improve_count += 1
    
                if no_improve_count >= patience:
                    print(f"Early stopping: No improvement in validation loss for {patience} epochs.")
                    break
    
            if epoch_time > time_threshold:
                print(f"Early stopping: Epoch time exceeded {time_threshold} seconds.")
                break
    
        return epoch_summary
        
    def predict(self, user_id, item_id):
        u = self.user_factors[user_id]
        p = self.common_item_factors[item_id]
        q_key = (user_id, item_id)
        q = self.personal_item_factors.get(q_key, np.zeros(self.latent_dim))
        return np.dot(u, p + q)

    def compute_rmse(self, data):
        mse = 0.0
        for user_id, item_id, rating in data:
            pred = self.predict(user_id, item_id)
            mse += (rating - pred) ** 2
        return np.sqrt(mse / len(data))

In [26]:
train_loader_type = "rs"
val_loader_type = "rs"
batch_size = 50
userprop = 0.6

train_df = pd.read_csv("dataset/ml100k_train_seed10.csv")
test_df = pd.read_csv("dataset/ml100k_test_seed10.csv")
train_df.head()

n_users = train_df['user_id'].nunique()
n_items = train_df['item_id'].nunique()
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)


Total User: 943
Total Item: 1633
